# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [8]:
from dotenv import load_dotenv
load_dotenv()



True

In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [10]:
import os
from glob import glob

price_data_path = os.getenv("PRICE_DATA")
file_paths = glob(os.path.join(price_data_path, "**", "*.parquet"), recursive=True)
print("Found files:", file_paths)

Found files: ['../../05_src/data/prices/BKTI/BKTI_2012/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_2012/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_2015/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_2015/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_2014/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_2014/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_2013/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_2013/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_1980/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_1980/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_1987/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_1987/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_1989/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_1989/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_1988/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_1988/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_1986/part.0.parquet', 

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [11]:
import os
from glob import glob
import dask.dataframe as dd

price_data_path = os.getenv("PRICE_DATA")
file_paths = glob(os.path.join(price_data_path, "**/*.parquet"), recursive=True)

ddf = dd.read_parquet(file_paths)

ddf.columns = [col.replace(" ", "_") for col in ddf.columns]
ddf = ddf.sort_values(["ticker", "Date"])

ddf["Close_lag_1"] = ddf.groupby("ticker")["Close"].shift(1, meta=('Close_lag_1', 'f8'))
ddf["Adj_Close_lag_1"] = ddf.groupby("ticker")["Adj_Close"].shift(1, meta=('Adj_Close_lag_1', 'f8'))

ddf["returns"] = (ddf["Close"] / ddf["Close_lag_1"]) - 1
ddf["hi_lo_range"] = ddf["High"] - ddf["Low"]

dd_feat = ddf



+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [24]:
pdf['returns'] = pdf.groupby('ticker')['Adj Close'].pct_change()
pdf['returns_ma_10'] = pdf.groupby('ticker')['returns'].transform(lambda x: x.rolling(window=10).mean())

print(pdf.head(10))

            Date       Open       High        Low      Close  Adj Close  \
38739 1999-11-18  32.546494  35.765381  28.612303  31.473534  27.068665   
38740 1999-11-19  30.713520  30.758226  28.478184  28.880543  24.838577   
38741 1999-11-22  29.551144  31.473534  28.657009  31.473534  27.068665   
38742 1999-11-23  30.400572  31.205294  28.612303  28.612303  24.607880   
38743 1999-11-24  28.701717  29.998211  28.612303  29.372318  25.261524   
38744 1999-11-26  29.238197  29.685265  29.148785  29.461731  25.338428   
38745 1999-11-29  29.327610  30.355865  29.014664  30.132332  25.915169   
38746 1999-11-30  30.042919  30.713520  29.282904  30.177038  25.953619   
38747 1999-12-01  30.177038  31.071173  29.953505  30.713520  26.415012   
38748 1999-12-02  31.294706  32.188843  30.892345  31.562946  27.145563   

           Volume source ticker  Year   returns  returns_ma_10  
38739  62546300.0  A.csv      A  1999       NaN            NaN  
38740  15234100.0  A.csv      A  1999 -0.082

In [25]:
import os
from glob import glob

price_data_path = os.getenv("PRICE_DATA")
file_paths = glob(os.path.join(price_data_path, "**", "*.parquet"), recursive=True)
print("Found files:", file_paths)

Found files: ['../../05_src/data/prices/BKTI/BKTI_2012/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_2012/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_2015/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_2015/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_2014/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_2014/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_2013/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_2013/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_1980/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_1980/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_1987/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_1987/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_1989/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_1989/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_1988/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_1988/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_1986/part.0.parquet', 

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

It’s not necessary.
For bigger data, better to do it directly in Dask to save memory and use parallel computing.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.